# Stage 5. Multi-Task Training and Model Comparison

Notebook ini melatih model multi-task dual-path memakai validasi silang lima-fold berbasis pasien, lalu membandingkan lima konfigurasi model untuk membuktikan pilihan desain secara empiris. Perbandingan mencakup jalur fitur (hand-crafted saja, deep embedding saja, atau keduanya) dan backbone deep embedding (ResNet18-CSA lawan MobileNetV3-CSA).

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd
import torch

from configs import paths
from src.common import features, manifest as manifest_utils, train
from src.sites.conjunctiva import data

print("device", "cuda" if torch.cuda.is_available() else "cpu")

device cuda


## Load Manifest and Stage 4 Features

Manifest dibangun ulang lalu diberi kolom fold untuk validasi silang lima-fold, stratifikasi per kombinasi dataset dan label anemik. Fitur hand-crafted dan deep embedding ResNet18-CSA dimuat dari hasil Stage 4.

In [2]:
output_dir = paths.outputs_dir("conjunctiva")
artifact_dir = paths.artifacts_dir("conjunctiva")

manifest = manifest_utils.assign_kfold(data.build_manifest(save=False), n_splits=5, seed=42)
handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
embeddings_resnet18 = np.load(output_dir / "deep_embeddings.npy")
embedding_uids_resnet18 = pd.read_csv(output_dir / "deep_embeddings_uids.csv")["uid"].tolist()

print("manifest", manifest.shape)
print("fold balance:")
print(manifest.groupby(["dataset", "anemic", "fold"]).size().unstack(fill_value=0).to_string())

Eyes-Defy Italy melewati 2 folder tanpa metadata atau file lengkap: [93, 95]
Eyes-Defy India melewati 1 folder tanpa metadata atau file lengkap: [7]
manifest (925, 14)
fold balance:
fold               0   1   2   3   4
dataset   anemic                    
cp_anemic 0       58  57  57  57  57
          1       85  85  85  85  84
eyes_defy 0       25  25  25  25  25
          1       18  18  18  18  18


## Sanity Check: Handcrafted Features with Classical SVM

Sebelum melatih model neural yang lebih kompleks, fitur hand-crafted diuji dengan SVM klasik pada CP-AnemiC mengikuti protokol Paper 1, sebagai bukti bahwa implementasi fitur sudah benar. Akurasi validasi silang seharusnya mendekati 0.849 yang dilaporkan pada literatur.

In [3]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cp_manifest = manifest[manifest["dataset"] == "cp_anemic"].reset_index(drop=True)
cp_handcrafted = handcrafted.set_index("uid").loc[cp_manifest["uid"]].reset_index()
X_sanity = cp_handcrafted[features.HANDCRAFTED_COLUMNS].to_numpy()
y_sanity = cp_manifest["anemic"].to_numpy()

sanity_pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
sanity_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.1]}
sanity_search = GridSearchCV(
    sanity_pipeline, sanity_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy",
)
sanity_search.fit(X_sanity, y_sanity)
print("best params", sanity_search.best_params_)
print("cross validated accuracy", round(sanity_search.best_score_, 4))
print("target dari Paper 1 (SVM tuned)", 0.849)

best params {'svc__C': 100, 'svc__gamma': 0.1}
cross validated accuracy 0.8225
target dari Paper 1 (SVM tuned) 0.849


## Extract MobileNetV3-CSA Embeddings

Backbone kedua diekstraksi untuk perbandingan langsung dengan ResNet18-CSA. MobileNetV3 lebih ringan dan relevan untuk deployment pada perangkat mobile atau edge.

In [4]:
mobilenet_backbone = features.EmbeddingBackbone(backbone_name="mobilenet_v3_small")
embeddings_mobilenet, embedding_uids_mobilenet = features.extract_deep_embeddings(manifest, model=mobilenet_backbone)
print("mobilenet embeddings shape", embeddings_mobilenet.shape)
print("order matches manifest", list(embedding_uids_mobilenet) == list(manifest["uid"]))

mobilenet embeddings shape (925, 256)
order matches manifest True


## Five Model Configurations

Kelima konfigurasi dilatih dengan protokol validasi silang lima-fold yang identik agar perbandingan adil. Konfigurasi keempat, Full Fusion dengan ResNet18-CSA, adalah kandidat model utama.

In [5]:
configurations = [
    {"name": "Path A only", "use_handcrafted": True, "use_deep": False, "embeddings": None, "embedding_uids": None},
    {"name": "Path B only (ResNet18-CSA)", "use_handcrafted": False, "use_deep": True,
     "embeddings": embeddings_resnet18, "embedding_uids": embedding_uids_resnet18},
    {"name": "Path B only (MobileNetV3-CSA)", "use_handcrafted": False, "use_deep": True,
     "embeddings": embeddings_mobilenet, "embedding_uids": embedding_uids_mobilenet},
    {"name": "Full Fusion (ResNet18-CSA)", "use_handcrafted": True, "use_deep": True,
     "embeddings": embeddings_resnet18, "embedding_uids": embedding_uids_resnet18},
    {"name": "Full Fusion (MobileNetV3-CSA)", "use_handcrafted": True, "use_deep": True,
     "embeddings": embeddings_mobilenet, "embedding_uids": embedding_uids_mobilenet},
]

## Train and Evaluate All Configurations

Bobot loss gabungan memakai nilai tetap, yaitu satu untuk regresi hemoglobin, satu untuk klasifikasi anemia, dan setengah untuk severity ordinal karena severity hanya tersedia pada subset data CP-AnemiC.

In [6]:
results = {}
comparison_rows = []
for configuration in configurations:
    result = train.run_kfold(
        manifest, handcrafted, configuration["embeddings"], configuration["embedding_uids"],
        n_splits=5, epochs=60, batch_size=64, loss_weights=(1.0, 1.0, 0.5),
        use_handcrafted=configuration["use_handcrafted"], use_deep=configuration["use_deep"],
    )
    results[configuration["name"]] = result
    metrics = result["fold_metrics"]
    comparison_rows.append({
        "configuration": configuration["name"],
        "mae_mean": metrics["mae"].mean(),
        "mae_std": metrics["mae"].std(),
        "accuracy_mean": metrics["accuracy"].mean(),
        "accuracy_std": metrics["accuracy"].std(),
        "severity_accuracy_mean": metrics["severity_accuracy"].mean(),
    })
    print("selesai", configuration["name"])

comparison_table = pd.DataFrame(comparison_rows)
print()
print(comparison_table.round(4).to_string(index=False))

selesai Path A only


selesai Path B only (ResNet18-CSA)


selesai Path B only (MobileNetV3-CSA)


selesai Full Fusion (ResNet18-CSA)


selesai Full Fusion (MobileNetV3-CSA)

                configuration  mae_mean  mae_std  accuracy_mean  accuracy_std  severity_accuracy_mean
                  Path A only    1.6384   0.1027         0.6433        0.0596                  0.3057
   Path B only (ResNet18-CSA)    1.6712   0.1295         0.6422        0.0218                  0.3211
Path B only (MobileNetV3-CSA)    1.6751   0.1061         0.6206        0.0582                  0.3267
   Full Fusion (ResNet18-CSA)    1.5873   0.0639         0.6670        0.0488                  0.3155
Full Fusion (MobileNetV3-CSA)    1.6463   0.0796         0.6433        0.0328                  0.3183


## Compare Against Literature Baselines

MAE hemoglobin pada Eyes-Defy pada literatur BPANet sekitar 1.212 g/dL, dan pada CP-AnemiC dengan backbone ViT sekitar 1.50 g/dL. Nilai ini menjadi tolok ukur validitas model yang dilatih di sini, meski dihitung pada seluruh dataset gabungan sehingga tidak sepenuhnya identik protokolnya.

In [7]:
for configuration_name, result in results.items():
    oof = result["oof"].merge(manifest[["uid", "dataset"]], on="uid")
    mae_by_dataset = oof.groupby("dataset").apply(lambda g: np.mean(np.abs(g["hb_pred"] - g["hb_true"])))
    print(configuration_name)
    print(mae_by_dataset.round(4).to_string())
    print()

Path A only
dataset
cp_anemic    1.7602
eyes_defy    1.2369

Path B only (ResNet18-CSA)
dataset
cp_anemic    1.7841
eyes_defy    1.2992

Path B only (MobileNetV3-CSA)
dataset
cp_anemic    1.7965
eyes_defy    1.2747

Full Fusion (ResNet18-CSA)
dataset
cp_anemic    1.6987
eyes_defy    1.2197

Full Fusion (MobileNetV3-CSA)
dataset
cp_anemic    1.7695
eyes_defy    1.2400



## Save Results

Prediksi out-of-fold dan metrik per fold untuk setiap konfigurasi disimpan agar dapat dipakai pada evaluasi mendalam di Stage 6. Checkpoint model per fold untuk konfigurasi Full Fusion ResNet18-CSA, sebagai kandidat utama, disimpan ke folder artifacts.

In [8]:
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)

for configuration_name, result in results.items():
    slug = configuration_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    result["oof"].to_csv(output_dir / f"multitask_oof_{slug}.csv", index=False)
    result["fold_metrics"].to_csv(output_dir / f"multitask_fold_metrics_{slug}.csv", index=False)

main_candidate = results["Full Fusion (ResNet18-CSA)"]
for fold_index, fold_model in enumerate(main_candidate["models"]):
    checkpoint_path = artifact_dir / f"multitask_full_fusion_resnet18_fold{fold_index}.pt"
    torch.save(fold_model.state_dict(), checkpoint_path)

print("saved comparison table and per-configuration results to", output_dir)
print("saved fold checkpoints for main candidate to", artifact_dir)

saved comparison table and per-configuration results to /home/praktikan/projects/Azril/hemavision/outputs/conjunctiva
saved fold checkpoints for main candidate to /home/praktikan/projects/Azril/hemavision/artifacts/conjunctiva


## Diagnose Per-Dataset Performance

Akurasi gabungan dapat menyembunyikan kondisi satu dataset yang sebenarnya collapse ke kelas mayoritas. Setiap konfigurasi diperiksa performanya secara terpisah pada CP-AnemiC dan Eyes-Defy.

In [9]:
for configuration_name, result in results.items():
    breakdown = train.evaluate_by_dataset(result["oof"], manifest)
    print(configuration_name)
    print(breakdown.round(4).to_string(index=False))
    print()

Path A only
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.7602    0.6042             0.3056
eyes_defy 215 1.2369    0.7721                NaN

Path B only (ResNet18-CSA)
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.7841    0.6014             0.3211
eyes_defy 215 1.2992    0.7767                NaN

Path B only (MobileNetV3-CSA)
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.7965    0.5831             0.3268
eyes_defy 215 1.2747    0.7442                NaN

Full Fusion (ResNet18-CSA)
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.6987    0.6437             0.3155
eyes_defy 215 1.2197    0.7442                NaN

Full Fusion (MobileNetV3-CSA)
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.7695    0.6070             0.3183
eyes_defy 215 1.2400    0.7628                NaN



## Hyperparameter Search with Optuna

Pencarian difokuskan pada konfigurasi Full Fusion ResNet18-CSA, kandidat utama, agar biaya komputasi terkendali. Objective yang diminimalkan menggabungkan MAE hemoglobin ternormalisasi dengan akurasi kasus terburuk antar dataset, sehingga solusi yang bagus di satu dataset namun collapse di dataset lain akan dihukum, bukan hanya mengejar akurasi gabungan.

In [10]:
import optuna

CLINICAL_MAE_THRESHOLD = 3.6


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 30, 120)
    weight_classification = trial.suggest_float("weight_classification", 0.5, 3.0)
    weight_severity = trial.suggest_float("weight_severity", 0.1, 1.0)
    focal_gamma = trial.suggest_float("focal_gamma", 0.5, 3.0)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    trunk_dim = trial.suggest_categorical("trunk_dim", [64, 128, 256])
    attention_dim = trial.suggest_categorical("attention_dim", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    result = train.run_kfold(
        manifest, handcrafted, embeddings_resnet18, embedding_uids_resnet18,
        n_splits=5, epochs=epochs, batch_size=batch_size, learning_rate=learning_rate,
        loss_weights=(1.0, weight_classification, weight_severity),
        trunk_dim=trunk_dim, attention_dim=attention_dim, dropout=dropout, focal_gamma=focal_gamma,
        use_handcrafted=True, use_deep=True,
    )
    mae_mean = float(result["fold_metrics"]["mae"].mean())
    breakdown = train.evaluate_by_dataset(result["oof"], manifest)
    worst_case_accuracy = float(breakdown["accuracy"].min())
    score = mae_mean / CLINICAL_MAE_THRESHOLD + (1.0 - worst_case_accuracy)

    trial.set_user_attr("mae_mean", mae_mean)
    trial.set_user_attr("worst_case_accuracy", worst_case_accuracy)
    return score


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("best value", round(study.best_value, 4))
print("best params", study.best_params)
print("best mae_mean", round(study.best_trial.user_attrs["mae_mean"], 4))
print("best worst_case_accuracy", round(study.best_trial.user_attrs["worst_case_accuracy"], 4))

[I 2026-07-22 13:39:50,423] A new study created in memory with name: no-name-572206d8-daa4-452c-a8a6-d929ef8bccdd


[I 2026-07-22 13:39:54,658] Trial 0 finished with value: 0.8699555013661093 and parameters: {'learning_rate': 0.00035161771311837217, 'epochs': 37, 'weight_classification': 2.561036778953905, 'weight_severity': 0.3185632377145394, 'focal_gamma': 2.7543571554793105, 'dropout': 0.4919239642067652, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 64}. Best is trial 0 with value: 0.8699555013661093.


[I 2026-07-22 13:40:00,698] Trial 1 finished with value: 0.7086516440007982 and parameters: {'learning_rate': 0.0008243326108019368, 'epochs': 108, 'weight_classification': 2.0472439627229484, 'weight_severity': 0.7747610941856381, 'focal_gamma': 2.9732523544049885, 'dropout': 0.35372879318097195, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 128}. Best is trial 1 with value: 0.7086516440007982.


[I 2026-07-22 13:40:08,551] Trial 2 finished with value: 0.764338255152456 and parameters: {'learning_rate': 0.00886866045140502, 'epochs': 36, 'weight_classification': 2.086004447146367, 'weight_severity': 0.5127036445090252, 'focal_gamma': 2.6533613598481662, 'dropout': 0.1470779296481587, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 32}. Best is trial 1 with value: 0.7086516440007982.


[I 2026-07-22 13:40:13,103] Trial 3 finished with value: 0.8425803854618461 and parameters: {'learning_rate': 0.0006731158113220066, 'epochs': 42, 'weight_classification': 1.723455670662468, 'weight_severity': 0.7696079917170686, 'focal_gamma': 1.3406667381384119, 'dropout': 0.41646845984694614, 'trunk_dim': 128, 'attention_dim': 128, 'batch_size': 64}. Best is trial 1 with value: 0.7086516440007982.


...

[I 2026-07-22 13:44:58,792] Trial 36 finished with value: 0.6895359060252977 and parameters: {'learning_rate': 0.008009824998068918, 'epochs': 109, 'weight_classification': 2.6369668210459483, 'weight_severity': 0.5031622025477991, 'focal_gamma': 2.1724054222868623, 'dropout': 0.22725797237421186, 'trunk_dim': 256, 'attention_dim': 64, 'batch_size': 128}. Best is trial 30 with value: 0.6371161349984102.


[I 2026-07-22 13:45:08,759] Trial 37 finished with value: 0.6366935974182284 and parameters: {'learning_rate': 0.003055737416493421, 'epochs': 84, 'weight_classification': 1.9482952299704612, 'weight_severity': 0.3896833153575715, 'focal_gamma': 2.0367179391695727, 'dropout': 0.3119141432082831, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 64}. Best is trial 37 with value: 0.6366935974182284.


[I 2026-07-22 13:45:16,489] Trial 38 finished with value: 0.7590276812723544 and parameters: {'learning_rate': 0.0021422528949997877, 'epochs': 68, 'weight_classification': 1.977639607139562, 'weight_severity': 0.38366016461247265, 'focal_gamma': 2.0180429828516835, 'dropout': 0.2842736309837054, 'trunk_dim': 64, 'attention_dim': 32, 'batch_size': 64}. Best is trial 37 with value: 0.6366935974182284.


[I 2026-07-22 13:45:25,729] Trial 39 finished with value: 0.6860494168151712 and parameters: {'learning_rate': 0.002992157204269129, 'epochs': 83, 'weight_classification': 1.8475333546483177, 'weight_severity': 0.2351552356263261, 'focal_gamma': 1.7797923149277395, 'dropout': 0.23808729444240254, 'trunk_dim': 128, 'attention_dim': 32, 'batch_size': 64}. Best is trial 37 with value: 0.6366935974182284.


best value 0.6367
best params {'learning_rate': 0.003055737416493421, 'epochs': 84, 'weight_classification': 1.9482952299704612, 'weight_severity': 0.3896833153575715, 'focal_gamma': 2.0367179391695727, 'dropout': 0.3119141432082831, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 64}
best mae_mean 1.5163
best worst_case_accuracy 0.7845


## Retrain with Best Hyperparameters

Full Fusion ResNet18-CSA dilatih ulang memakai hyperparameter terbaik hasil pencarian Optuna, lalu dibandingkan dengan hasil sebelum tuning baik secara gabungan maupun per dataset.

In [11]:
best_params = study.best_params
tuned_result = train.run_kfold(
    manifest, handcrafted, embeddings_resnet18, embedding_uids_resnet18,
    n_splits=5,
    epochs=best_params["epochs"], batch_size=best_params["batch_size"], learning_rate=best_params["learning_rate"],
    loss_weights=(1.0, best_params["weight_classification"], best_params["weight_severity"]),
    trunk_dim=best_params["trunk_dim"], attention_dim=best_params["attention_dim"],
    dropout=best_params["dropout"], focal_gamma=best_params["focal_gamma"],
    use_handcrafted=True, use_deep=True,
)
tuned_name = "Full Fusion (ResNet18-CSA, tuned)"
results[tuned_name] = tuned_result

print(tuned_result["fold_metrics"].round(4).to_string(index=False))
print()
print("before tuning vs after tuning, per dataset:")
print("Full Fusion (ResNet18-CSA), before tuning")
print(train.evaluate_by_dataset(results["Full Fusion (ResNet18-CSA)"]["oof"], manifest).round(4).to_string(index=False))
print()
print(tuned_name)
print(train.evaluate_by_dataset(tuned_result["oof"], manifest).round(4).to_string(index=False))

 fold  n_val    mae   rmse  accuracy  severity_accuracy
    0    186 1.6334 2.2282    0.7688             0.3986
    1    185 1.5416 1.9650    0.6811             0.4085
    2    185 1.3798 1.8138    0.8000             0.5423
    3    185 1.7641 2.2583    0.6811             0.2958
    4    184 1.4500 1.9033    0.7337             0.4326

before tuning vs after tuning, per dataset:
Full Fusion (ResNet18-CSA), before tuning
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.6987    0.6437             0.3155
eyes_defy 215 1.2197    0.7442                NaN

Full Fusion (ResNet18-CSA, tuned)
  dataset   n    mae  accuracy  severity_accuracy
cp_anemic 710 1.6461    0.7423             0.4155
eyes_defy 215 1.2497    0.7023                NaN


## Updated Comparison and Save

In [12]:
updated_rows = list(comparison_rows)
tuned_metrics = tuned_result["fold_metrics"]
updated_rows.append({
    "configuration": tuned_name,
    "mae_mean": tuned_metrics["mae"].mean(),
    "mae_std": tuned_metrics["mae"].std(),
    "accuracy_mean": tuned_metrics["accuracy"].mean(),
    "accuracy_std": tuned_metrics["accuracy"].std(),
    "severity_accuracy_mean": tuned_metrics["severity_accuracy"].mean(),
})
updated_comparison_table = pd.DataFrame(updated_rows)
print(updated_comparison_table.round(4).to_string(index=False))

updated_comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)
tuned_result["oof"].to_csv(output_dir / "multitask_oof_full_fusion_resnet18_csa_tuned.csv", index=False)
tuned_result["fold_metrics"].to_csv(output_dir / "multitask_fold_metrics_full_fusion_resnet18_csa_tuned.csv", index=False)

import json
with open(output_dir / "multitask_optuna_best_params.json", "w") as handle:
    json.dump(
        {"best_params": study.best_params, "best_value": study.best_value,
         "mae_mean": study.best_trial.user_attrs["mae_mean"],
         "worst_case_accuracy": study.best_trial.user_attrs["worst_case_accuracy"]},
        handle, indent=2,
    )

if tuned_metrics["mae"].mean() <= comparison_table.loc[
    comparison_table["configuration"] == "Full Fusion (ResNet18-CSA)", "mae_mean"
].iloc[0]:
    for fold_index, fold_model in enumerate(tuned_result["models"]):
        checkpoint_path = artifact_dir / f"multitask_full_fusion_resnet18_tuned_fold{fold_index}.pt"
        torch.save(fold_model.state_dict(), checkpoint_path)
    print("tuned checkpoints saved, MAE membaik atau setara dibanding sebelum tuning")
else:
    print("tuned checkpoints tidak menimpa, MAE sebelum tuning masih lebih baik")

                    configuration  mae_mean  mae_std  accuracy_mean  accuracy_std  severity_accuracy_mean
                      Path A only    1.6384   0.1027         0.6433        0.0596                  0.3057
       Path B only (ResNet18-CSA)    1.6712   0.1295         0.6422        0.0218                  0.3211
    Path B only (MobileNetV3-CSA)    1.6751   0.1061         0.6206        0.0582                  0.3267
       Full Fusion (ResNet18-CSA)    1.5873   0.0639         0.6670        0.0488                  0.3155
    Full Fusion (MobileNetV3-CSA)    1.6463   0.0796         0.6433        0.0328                  0.3183
Full Fusion (ResNet18-CSA, tuned)    1.5538   0.1515         0.7329        0.0528                  0.4155
tuned checkpoints saved, MAE membaik atau setara dibanding sebelum tuning
